In [ ]:
%pip install -q "transformers>=4.46,<6" "accelerate>=1.0,<2"


In [ ]:
import copy
import csv
import os
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from bs4 import BeautifulSoup
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Kaggle 上写 /kaggle/working，本地写 ./outputs
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 各实验的验证集 AUC，训练单元会往里写，最后一节汇总
RESULTS = {}

print('PyTorch 版本：', torch.__version__)
print('训练设备：', device)
print('输出目录：', OUTPUT_DIR)


In [ ]:
# 数据文件可能在这几个位置：环境变量、Kaggle 输入目录、当前目录
DATA_DIR_CANDIDATES = [
    os.environ.get('WORD2VEC_DATA_DIR', ''),
    '/kaggle/input/competitions/word2vec-nlp-tutorial',
    '/kaggle/input/word2vec-nlp-tutorial',
    os.path.abspath('.'),
    os.path.abspath('./data'),
]

TRAIN_FILES = ('labeledTrainData.tsv', 'labeledTrainData.tsv.zip')
TEST_FILES = ('testData.tsv', 'testData.tsv.zip')


def find_data_file(filenames):
    """按候选目录顺序返回第一个存在的数据文件。"""
    for directory in DATA_DIR_CANDIDATES:
        if not directory:
            continue
        for filename in filenames:
            path = os.path.join(directory, filename)
            if os.path.exists(path):
                return path
    return None


def read_review_tsv(path, num_columns):
    """读带 review 列的 TSV；引号不规范时退回 QUOTE_NONE（等价教程里的 quoting=3）。"""
    data = pd.read_csv(path, sep='\t')
    if data.shape[1] != num_columns:
        data = pd.read_csv(path, sep='\t', quoting=csv.QUOTE_NONE)
    return data


train_path = find_data_file(TRAIN_FILES)
test_path = find_data_file(TEST_FILES)

if train_path is None or test_path is None:
    # 没挂载数据集时用 kagglehub 现场下载
    try:
        import kagglehub

        dataset_dir = kagglehub.competition_download('word2vec-nlp-tutorial')
        DATA_DIR_CANDIDATES.insert(0, dataset_dir)
        train_path = find_data_file(TRAIN_FILES)
        test_path = find_data_file(TEST_FILES)
    except Exception as error:
        print('kagglehub 下载失败：', error)

assert train_path is not None, '没找到 labeledTrainData.tsv(.zip)，请放到当前目录或设置 WORD2VEC_DATA_DIR'
assert test_path is not None, '没找到 testData.tsv(.zip)，请放到当前目录或设置 WORD2VEC_DATA_DIR'

print('训练集文件：', train_path)
print('测试集文件：', test_path)

train = read_review_tsv(train_path, 3)
test = read_review_tsv(test_path, 2)

assert list(train.columns) == ['id', 'sentiment', 'review'], train.columns
assert list(test.columns) == ['id', 'review'], test.columns
assert len(test) == test['id'].nunique(), '测试集 id 有重复，读入可能出错'
assert train['sentiment'].isin([0, 1]).all(), '标签不是 0/1'

print('训练集：', train.shape)
print('测试集：', test.shape)
print(train.head())


In [ ]:
MAX_VOCAB = 50000


def tokenize_review(text):
    """去 HTML 标签，转小写，只保留英文字母。"""
    text = BeautifulSoup(str(text), 'html.parser').get_text(' ')
    return re.findall(r'[a-z]+', text.lower())


tokenized_train = [tokenize_review(review) for review in train['review'].values]
tokenized_test = [tokenize_review(review) for review in test['review'].values]

review_lengths = [len(tokens) for tokens in tokenized_train]
print('评论长度统计：')
print(pd.Series(review_lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

# 原实验：第 95 百分位，限制在 100 ~ 500
MAX_LEN = int(np.percentile(review_lengths, 95))
MAX_LEN = min(max(MAX_LEN, 100), 500)
print('MAX_LEN =', MAX_LEN)

# 按训练集词频降序建词表，格式和原 notebook 一致
word_counter = Counter(word for tokens in tokenized_train for word in tokens)
vocab_words = [word for word, _ in word_counter.most_common(MAX_VOCAB)]

word_to_idx = {word: index + 2 for index, word in enumerate(vocab_words)}
VOCAB_SIZE = len(vocab_words) + 2

print('词表大小：', VOCAB_SIZE)
print('最高频词：', vocab_words[:10])


In [ ]:
def encode_tokens(tokens, max_len=None):
    """词 -> id，截断 / 补齐到 max_len，同时返回真实长度。"""

    if max_len is None:
        max_len = MAX_LEN

    ids = [word_to_idx.get(word, 1) for word in tokens]

    # 防止空评论
    if len(ids) == 0:
        ids = [1]

    ids = ids[:max_len]
    length = len(ids)
    ids += [0] * (max_len - length)

    return ids, length


def build_sequences(tokenized_reviews):
    sequences = []
    lengths = []
    for tokens in tokenized_reviews:
        ids, length = encode_tokens(tokens)
        sequences.append(ids)
        lengths.append(length)
    return (
        np.array(sequences, dtype=np.int64),
        np.array(lengths, dtype=np.int64)
    )


X_all, lengths_all = build_sequences(tokenized_train)
X_test, len_test = build_sequences(tokenized_test)
y_all = train['sentiment'].values.astype(np.float32)

print('训练序列矩阵：', X_all.shape)
print('训练长度矩阵：', lengths_all.shape)
print('测试序列矩阵：', X_test.shape)
print('标签矩阵：', y_all.shape)

total_words = sum(len(tokens) for tokens in tokenized_train)
known_words = sum(
    1
    for tokens in tokenized_train
    for word in tokens
    if word in word_to_idx
)
print('词表覆盖率：', known_words / max(total_words, 1))


In [ ]:
(
    X_train, X_val,
    len_train, len_val,
    y_train, y_val
) = train_test_split(
    X_all,
    lengths_all,
    y_all,
    test_size=0.2,
    random_state=SEED,
    stratify=y_all
)

print('训练集：', X_train.shape)
print('验证集：', X_val.shape)

BATCH_SIZE = 64

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.long),
    torch.tensor(len_train, dtype=torch.long),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.long),
    torch.tensor(len_val, dtype=torch.long),
    torch.tensor(y_val, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.long),
    torch.tensor(len_test, dtype=torch.long)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print('训练批次数：', len(train_loader))
print('验证批次数：', len(val_loader))
print('测试批次数：', len(test_loader))


In [ ]:
criterion = nn.BCEWithLogitsLoss()


def evaluate_model(net, loader):

    net.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():

        for x, lengths, labels in loader:

            x = x.to(device)
            labels = labels.to(device)

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)

            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    auc = roc_auc_score(all_labels, all_probs)

    return avg_loss, auc


def fit_model(net, train_loader, val_loader, epochs=10, lr=0.001, patience=3):

    optimizer = torch.optim.AdamW(
        net.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    best_auc = -1
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):

        net.train()

        total_loss = 0

        for x, lengths, labels in train_loader:

            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            loss.backward()

            # 防止梯度过大
            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(labels)

        train_loss = total_loss / len(train_loader.dataset)

        val_loss, val_auc = evaluate_model(net, val_loader)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

        if val_auc > best_auc:

            best_auc = val_auc

            best_state = copy.deepcopy(net.state_dict())

            wait = 0

        else:

            wait += 1

        if wait >= patience:

            print('Early Stopping')
            break

    net.load_state_dict(best_state)

    return net, best_auc


def predict_test(net, test_loader):

    net.eval()

    predictions = []

    with torch.no_grad():

        for x, lengths in test_loader:

            x = x.to(device)

            logits = net(x, lengths)

            probs = torch.sigmoid(logits)

            predictions.extend(probs.cpu().numpy())

    return np.array(predictions)


def save_submission(predictions, filename):

    submission = pd.DataFrame({
        'id': test['id'].values,
        'sentiment': np.asarray(predictions, dtype=np.float32)
    })

    # 检查提交文件
    assert len(submission) == len(test)
    assert submission['id'].is_unique
    assert submission['id'].tolist() == test['id'].tolist()
    assert np.isfinite(submission['sentiment']).all()
    assert submission['sentiment'].between(0, 1).all()

    path = os.path.join(OUTPUT_DIR, filename)

    submission.to_csv(path, index=False)

    print('文件已生成：', path)
    print('文件形状：', submission.shape)
    print(submission.head())

    return submission


In [ ]:
from transformers import BertConfig, BertModel


class BertScratchClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_dim=128,
        num_layers=2,
        nhead=4,
        max_len=512
    ):
        super().__init__()

        # 词表外再加两个特殊 token：[CLS] 和 [SEP]
        self.cls_id = vocab_size
        self.sep_id = vocab_size + 1

        config = BertConfig(
            vocab_size=vocab_size + 2,
            hidden_size=hidden_dim,
            num_hidden_layers=num_layers,
            num_attention_heads=nhead,
            intermediate_size=hidden_dim * 4,
            hidden_act='gelu',
            hidden_dropout_prob=0.3,
            attention_probs_dropout_prob=0.1,
            max_position_embeddings=max_len + 2,
            pad_token_id=0
        )
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):

        lengths = lengths.to(x.device)

        # [CLS] + 词 + [SEP]，其余留 0（PAD）
        cls_tokens = x.new_full((x.size(0), 1), self.cls_id)
        input_ids = torch.cat(
            [cls_tokens, x, x.new_zeros(x.size(0), 1)],
            dim=1
        )
        input_ids.scatter_(1, (lengths + 1).unsqueeze(1), self.sep_id)

        positions = torch.arange(input_ids.size(1), device=x.device)
        attention_mask = positions.unsqueeze(0) < (lengths + 2).unsqueeze(1)

        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask.long(),
            return_dict=True
        )
        cls_output = output.last_hidden_state[:, 0]

        return self.fc(self.dropout(cls_output)).squeeze(1)


In [ ]:
print('开始训练 BERT Scratch')

bert_scratch_model = BertScratchClassifier(
    VOCAB_SIZE,
    hidden_dim=128,
    num_layers=2,
    nhead=4,
    max_len=MAX_LEN
).to(device)

bert_scratch_model, bert_scratch_best_auc = fit_model(
    bert_scratch_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.0003,
    patience=3
)

print('BERT Scratch 最佳验证集 AUC：', bert_scratch_best_auc)

torch.save(
    bert_scratch_model.state_dict(),
    os.path.join(OUTPUT_DIR, 'bert_scratch_best.pth')
)

bert_scratch_predictions = predict_test(bert_scratch_model, test_loader)

bert_scratch_submission = save_submission(
    bert_scratch_predictions,
    'submission_bert_scratch.csv'
)

RESULTS['BERT Scratch'] = bert_scratch_best_auc


In [ ]:
from transformers import AutoTokenizer, BertModel

BERT_NATIVE_NAME = 'google-bert/bert-base-uncased'
BERT_NATIVE_MAX_LEN = 256
BERT_NATIVE_BATCH_SIZE = 8

bert_native_tokenizer = AutoTokenizer.from_pretrained(BERT_NATIVE_NAME)


def encode_bert_reviews(reviews):
    texts = [
        BeautifulSoup(str(review), 'html.parser').get_text(' ')
        for review in reviews
    ]
    encoded = bert_native_tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=BERT_NATIVE_MAX_LEN,
        return_token_type_ids=False,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask'].sum(dim=1)


bert_native_ids, bert_native_lengths = encode_bert_reviews(train['review'])
bert_native_test_ids, bert_native_test_lengths = encode_bert_reviews(test['review'])
bert_native_labels = torch.tensor(train['sentiment'].values, dtype=torch.float32)

bert_native_train_idx, bert_native_val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=SEED,
    stratify=train['sentiment'].values
)

bert_native_train_dataset = TensorDataset(
    bert_native_ids[bert_native_train_idx],
    bert_native_lengths[bert_native_train_idx],
    bert_native_labels[bert_native_train_idx]
)
bert_native_val_dataset = TensorDataset(
    bert_native_ids[bert_native_val_idx],
    bert_native_lengths[bert_native_val_idx],
    bert_native_labels[bert_native_val_idx]
)
bert_native_test_dataset = TensorDataset(
    bert_native_test_ids,
    bert_native_test_lengths
)

bert_native_train_loader = DataLoader(
    bert_native_train_dataset,
    batch_size=BERT_NATIVE_BATCH_SIZE,
    shuffle=True
)
bert_native_val_loader = DataLoader(
    bert_native_val_dataset,
    batch_size=BERT_NATIVE_BATCH_SIZE,
    shuffle=False
)
bert_native_test_loader = DataLoader(
    bert_native_test_dataset,
    batch_size=BERT_NATIVE_BATCH_SIZE,
    shuffle=False
)

print('训练集：', len(bert_native_train_dataset))
print('验证集：', len(bert_native_val_dataset))
print('测试集：', len(bert_native_test_dataset))


In [ ]:
class BertNativeClassifier(nn.Module):

    def __init__(self, model_name=BERT_NATIVE_NAME):
        super().__init__()
        self.bert = BertModel.from_pretrained(
            model_name,
            add_pooling_layer=False
        )
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, x, lengths):

        # 分词器已经 padding 过，这里按真实长度重建 attention mask
        positions = torch.arange(x.size(1), device=x.device)
        attention_mask = positions.unsqueeze(0) < lengths.to(x.device).unsqueeze(1)

        output = self.bert(
            input_ids=x,
            attention_mask=attention_mask.long(),
            return_dict=True
        )
        cls_output = output.last_hidden_state[:, 0]

        return self.fc(self.dropout(cls_output)).squeeze(1)


In [ ]:
print('开始训练 BERT Native')

bert_native_model = BertNativeClassifier().to(device)

bert_native_model, bert_native_best_auc = fit_model(
    bert_native_model,
    bert_native_train_loader,
    bert_native_val_loader,
    epochs=3,
    lr=2e-5,
    patience=2
)

print('BERT Native 最佳验证集 AUC：', bert_native_best_auc)

torch.save(
    bert_native_model.state_dict(),
    os.path.join(OUTPUT_DIR, 'bert_native_best.pth')
)

bert_native_predictions = predict_test(
    bert_native_model,
    bert_native_test_loader
)

bert_native_submission = save_submission(
    bert_native_predictions,
    'submission_bert_native.csv'
)

RESULTS['BERT Native'] = bert_native_best_auc


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
    EarlyStoppingCallback, set_seed
)

BERT_TRAINER_NAME = 'google-bert/bert-base-uncased'
BERT_TRAINER_MAX_LEN = 256
BERT_TRAINER_BATCH_SIZE = 8

bert_trainer_tokenizer = AutoTokenizer.from_pretrained(BERT_TRAINER_NAME)


class BertTrainerDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = int(self.labels[index])
        return item


def encode_bert_trainer_reviews(reviews):
    texts = [
        BeautifulSoup(str(review), 'html.parser').get_text(' ')
        for review in reviews
    ]
    return bert_trainer_tokenizer(
        texts,
        truncation=True,
        max_length=BERT_TRAINER_MAX_LEN
    )


bert_trainer_train_idx, bert_trainer_val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=SEED,
    stratify=train['sentiment'].values
)
bert_trainer_labels = train['sentiment'].values

bert_trainer_train_dataset = BertTrainerDataset(
    encode_bert_trainer_reviews(train['review'].iloc[bert_trainer_train_idx]),
    bert_trainer_labels[bert_trainer_train_idx]
)
bert_trainer_val_dataset = BertTrainerDataset(
    encode_bert_trainer_reviews(train['review'].iloc[bert_trainer_val_idx]),
    bert_trainer_labels[bert_trainer_val_idx]
)
bert_trainer_test_dataset = BertTrainerDataset(
    encode_bert_trainer_reviews(test['review'])
)

print('训练集：', len(bert_trainer_train_dataset))
print('验证集：', len(bert_trainer_val_dataset))
print('测试集：', len(bert_trainer_test_dataset))


In [ ]:
def compute_bert_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    return {'auc': roc_auc_score(labels, probabilities)}


set_seed(SEED)
bert_trainer_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_TRAINER_NAME,
    num_labels=2
)

bert_trainer_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, 'bert_trainer_checkpoints'),
    learning_rate=2e-5,
    per_device_train_batch_size=BERT_TRAINER_BATCH_SIZE,
    per_device_eval_batch_size=BERT_TRAINER_BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    disable_tqdm=True,
    seed=SEED
)

bert_trainer = Trainer(
    model=bert_trainer_model,
    args=bert_trainer_args,
    train_dataset=bert_trainer_train_dataset,
    eval_dataset=bert_trainer_val_dataset,
    processing_class=bert_trainer_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=bert_trainer_tokenizer,
        pad_to_multiple_of=8
    ),
    compute_metrics=compute_bert_trainer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
bert_trainer.train()
print('BERT Trainer 最佳验证集 AUC：', bert_trainer.state.best_metric)

bert_trainer.save_model(os.path.join(OUTPUT_DIR, 'bert_trainer_best'))
bert_trainer_tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'bert_trainer_best'))


In [ ]:
bert_trainer_test_output = bert_trainer.predict(bert_trainer_test_dataset)
bert_trainer_predictions = torch.softmax(
    torch.from_numpy(bert_trainer_test_output.predictions), dim=-1
).numpy()[:, 1]

bert_trainer_submission = save_submission(
    bert_trainer_predictions,
    'submission_bert_trainer.csv'
)

RESULTS['BERT Trainer'] = float(bert_trainer.state.best_metric)
